# Batch Collection: All Models

This notebook runs the full 42-question bank across all models in sequence, handling errors and rate limits, and saves all responses to `data/responses/`.

**Before running:** ensure all API keys are set and you have completed setup in [Environment Setup](setup.md).

In [ ]:
import anthropic
import openai
import google.generativeai as genai
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("data/responses")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPERATURE = 0.0
MAX_TOKENS = 1024

SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

questions = pd.read_csv("../figures/question_bank.csv")
print(f"Loaded {len(questions)} questions across {questions['figure'].nunique()} figures")

In [ ]:
# ── Client initialisation ─────────────────────────────────────────────────────
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
openai_client    = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
xai_client       = openai.OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
gemini_model = genai.GenerativeModel(
    model_name="gemini-1.5-pro",
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS),
)

In [ ]:
# ── Model query functions ─────────────────────────────────────────────────────

def query_claude(q):
    msg = anthropic_client.messages.create(
        model="claude-opus-4-6", max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
        system=SYSTEM_PROMPT, messages=[{"role": "user", "content": q}],
    )
    return msg.content[0].text, msg.usage.input_tokens + msg.usage.output_tokens, msg.model

def query_openai(q):
    r = openai_client.chat.completions.create(
        model="gpt-4o", temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
    )
    return r.choices[0].message.content, r.usage.total_tokens, r.model

def query_grok(q):
    r = xai_client.chat.completions.create(
        model="grok-2", temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
    )
    return r.choices[0].message.content, r.usage.total_tokens, r.model

def query_gemini(q):
    r = gemini_model.generate_content(q)
    return r.text, r.usage_metadata.total_token_count, "gemini-1.5-pro"

MODELS = [
    ("claude",  query_claude,  0.5),
    ("openai",  query_openai,  0.5),
    ("grok",    query_grok,    0.5),
    ("gemini",  query_gemini,  1.0),
]

In [ ]:
# ── Batch run ─────────────────────────────────────────────────────────────────
summary = {}

for model_name, query_fn, sleep_s in MODELS:
    out_path = OUTPUT_DIR / f"{model_name}.jsonl"
    counts = {"ok": 0, "error": 0}
    print(f"\n{'='*50}\nCollecting: {model_name}\n{'='*50}")

    with open(out_path, "w") as f:
        for _, row in questions.iterrows():
            print(f"  {row['question_id']}...", end=" ")
            try:
                text, tokens, version = query_fn(row["question"])
                record = {
                    "question_id": row["question_id"],
                    "figure": row["figure"],
                    "model": model_name,
                    "model_version": version,
                    "prompt": row["question"],
                    "response": text,
                    "temperature": TEMPERATURE,
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "tokens_used": tokens,
                }
                f.write(json.dumps(record) + "\n")
                counts["ok"] += 1
                print(f"ok ({tokens} tok)")
            except Exception as e:
                counts["error"] += 1
                print(f"ERROR: {e}")
            time.sleep(sleep_s)

    summary[model_name] = counts
    print(f"  Saved to {out_path} — {counts}")

print("\n=== Batch complete ===")
for m, c in summary.items():
    print(f"  {m}: {c['ok']} ok, {c['error']} errors")